# Model Playground

## Setup

In [11]:
from __future__ import annotations

import os
import shutil
import subprocess
import sys
from pathlib import Path

# Use the local repository instead of Google Drive.
REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "train_headless.py").exists():
    alt_root = REPO_ROOT.parent
    if (alt_root / "train_headless.py").exists():
        REPO_ROOT = alt_root.resolve()
    else:
        raise FileNotFoundError(
            f"No repo checkout found locally under {REPO_ROOT}. Start the notebook from the repository root or change directories to the repo root."
        )

os.environ["JAX_PLATFORM_NAME"] = os.environ.get("QUADRUPED_NOTEBOOK_DEVICE", os.environ.get("JAX_PLATFORM_NAME", "auto"))
DEVICE_MODE = os.environ["JAX_PLATFORM_NAME"].lower()


def _gpu_available() -> bool:
    if shutil.which("nvidia-smi") is None:
        return False
    result = subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True)
    return result.returncode == 0 and bool(result.stdout.strip())


def _tpu_available() -> bool:
    return any(os.environ.get(name) for name in ("COLAB_TPU_ADDR", "TPU_NAME"))

if DEVICE_MODE == "auto":
    DEVICE_MODE = "gpu" if _gpu_available() else "cpu"
elif DEVICE_MODE not in {"cpu", "gpu", "tpu"}:
    raise ValueError("QUADRUPED_NOTEBOOK_DEVICE must be auto, cpu, gpu, or tpu.")
elif DEVICE_MODE == "gpu" and not _gpu_available():
    raise RuntimeError("GPU requested but nvidia-smi did not report one.")
elif DEVICE_MODE == "tpu" and not _tpu_available():
    raise RuntimeError("TPU requested but no Colab TPU environment was detected.")

os.environ["JAX_PLATFORM_NAME"] = DEVICE_MODE
sys.path = [entry for entry in sys.path if "multi-brain-quadruped-sim" not in entry]
sys.path.insert(0, str(REPO_ROOT))

{"repo": str(REPO_ROOT), "jax_platform": DEVICE_MODE}


{'repo': '/Users/monicagraham/Desktop/GitHub/multi-brain-quadruped-sim',
 'jax_platform': 'cpu'}

In [19]:
# Fix sys.path to include venv site-packages
import sys
from pathlib import Path

venv_site_packages = Path(sys.prefix) / "lib" / f"python{sys.version_info.major}.{sys.version_info.minor}" / "site-packages"
if str(venv_site_packages) not in sys.path:
    sys.path.insert(0, str(venv_site_packages))

# Now yaml should work
import yaml
print(f"✓ yaml version {yaml.__version__} loaded from {Path(yaml.__file__).parent}")


✓ yaml version 6.0.2 loaded from /Users/monicagraham/Desktop/GitHub/multi-brain-quadruped-sim/.venv/lib/python3.13/site-packages/yaml


## Imports

In [20]:
from dataclasses import replace

from brains.config import load_runtime_spec
from brains.harnesses import DirectionHarness, HeadCameraHarness
from brains.jax_trainer import ESTrainer, PARAM_COUNT, apply_runtime_spec
from brains.models import ModelDefinition, list_model_definitions, register_model_definition
from brains.runtime import create_model_run_paths, discover_model_artifacts, write_model_manifest

import brains

module_path = Path(brains.__file__).resolve()
if str(REPO_ROOT.resolve()) not in str(module_path):
    raise RuntimeError(f"Imported brains from {module_path}, expected it under {REPO_ROOT}")

{"imported_from": str(module_path), "param_count": PARAM_COUNT}


{'imported_from': '/Users/monicagraham/Desktop/GitHub/multi-brain-quadruped-sim/brains/__init__.py',
 'param_count': 48516}

## Model

In [21]:
MODEL_TYPE = "notebook_shared_trunk_v1"
LOG_ID = "notebook_demo_001"
SEED = 7

definition = ModelDefinition(
    type=MODEL_TYPE,
    architecture="shared_trunk_motor_lanes",
    trainer="openai_es",
    input_size=48,
    output_size=4,
    parameter_count=PARAM_COUNT,
    description="Local notebook variant of the current shared trunk ES trainer.",
)
register_model_definition(definition, persist=True)

spec = load_runtime_spec(REPO_ROOT / "configs" / "smoke.yaml")
spec = replace(
    spec,
    name="notebook-smoke",
    model=replace(
        spec.model,
        type=MODEL_TYPE,
        architecture=definition.architecture,
        trainer=definition.trainer,
        description=definition.description,
    ),
)
apply_runtime_spec(spec)

[model.to_dict() for model in list_model_definitions()]


[{'type': 'shared_trunk_es',
  'architecture': 'shared_trunk_motor_lanes',
  'trainer': 'openai_es',
  'input_size': 48,
  'output_size': 4,
  'parameter_count': 48516,
  'description': 'Current JAX policy vector with shared trunk and per-motor lanes.'},
 {'type': 'notebook_shared_trunk_v1',
  'architecture': 'shared_trunk_motor_lanes',
  'trainer': 'openai_es',
  'input_size': 48,
  'output_size': 4,
  'parameter_count': 48516,
  'description': 'Local notebook variant of the current shared trunk ES trainer.'}]

## Train

In [28]:
RUN_GENERATIONS = 100

trainer = ESTrainer(seed=SEED, spec=spec, model_id=f"{MODEL_TYPE}_{LOG_ID}", log_id=LOG_ID)
for i in range(RUN_GENERATIONS):
    trainer.run_generation()
    print(i)

{
    "model_type": spec.model.type,
    "generation": trainer.state.generation,
    "mean_reward": trainer.state.mean_reward,
    "best_reward": trainer.state.best_reward,
    "param_count": trainer.param_count,
}

0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50
51
52
53
54
55
56
57
58
59
60
61
62
63
64
65
66
67
68
69
70
71
72
73
74
75
76
77
78
79
80
81
82
83
84
85
86
87
88
89
90
91
92
93
94
95
96
97
98
99


{'model_type': 'notebook_shared_trunk_v1',
 'generation': 100,
 'mean_reward': -9.155000686645508,
 'best_reward': 18.895163845452686,
 'param_count': 48516}

## Save

In [29]:
paths = create_model_run_paths(REPO_ROOT / "checkpoints", MODEL_TYPE, LOG_ID)
trainer.model_id = paths.id
trainer.log_id = paths.log_id
saved_path = trainer.save_checkpoint(paths.latest_path)
write_model_manifest(
    paths,
    spec,
    saved_path,
    generation=trainer.state.generation,
    best_reward=trainer.state.best_reward,
    mean_reward=trainer.state.mean_reward,
)

{
    "model_id": paths.id,
    "latest": str(paths.latest_path),
    "manifest": str(paths.manifest_path),
    "visible_to_viewer": [artifact.id for artifact in discover_model_artifacts(REPO_ROOT / "checkpoints")],
}


{'model_id': 'notebook_shared_trunk_v1_notebook_demo_001',
 'latest': '/Users/monicagraham/Desktop/GitHub/multi-brain-quadruped-sim/checkpoints/notebook_shared_trunk_v1_notebook_demo_001/latest.npz',
 'manifest': '/Users/monicagraham/Desktop/GitHub/multi-brain-quadruped-sim/checkpoints/notebook_shared_trunk_v1_notebook_demo_001/manifest.json',
 'visible_to_viewer': ['notebook_shared_trunk_v1_notebook_demo_001']}

## Direction Harness

In [24]:
direction_harness = DirectionHarness(spec)
plan = direction_harness.compile("trot forward then turn left")

{
    "available_options": [option.name for option in direction_harness.available_options()],
    "plan": [command.option for command in plan.commands],
    "first_target_velocity": direction_harness.target_velocity(plan.commands[0].option, 0.25).tolist(),
}

{'available_options': ['stand',
  'trot',
  'turn_left',
  'turn_right',
  'back_up',
  'stop'],
 'plan': ['turn_left', 'trot'],
 'first_target_velocity': [0.4903096854686737,
  -0.4903096854686737,
  -1.6343656778335571,
  1.6343656778335571]}

## Camera Harness

In [25]:
camera_harness = HeadCameraHarness(spec)
camera_xml = camera_harness.build_camera_xml()

{
    "camera_name": camera_harness.camera.name,
    "image_shape": [camera_harness.camera.height, camera_harness.camera.width, 3],
    "camera_xml_contains_head_camera": "head_camera" in camera_xml,
}

{'camera_name': 'head_camera',
 'image_shape': [120, 160, 3],
 'camera_xml_contains_head_camera': True}